# LangChain Agent with a Self-Hosted LLM on Azure Container Apps

This notebook demonstrates how to build an **AI agent** using **LangChain** and **LangGraph** that connects to a self-hosted Gemma 4 model running on Azure Container Apps (ACA) with vLLM.

Since vLLM exposes an **OpenAI-compatible API**, we use LangChain's `ChatOpenAI` class to connect — no Azure OpenAI resource required.

## Key Concepts

1. **OpenAI** — The OpenAI-compatible API spec that vLLM implements, which LangChain can interface with.
2. **ChatOpenAI** — LangChain's chat model wrapper, pointed at our custom vLLM endpoint.
3. **Tools** — Python functions decorated with `@tool` that the agent can invoke.
4. **ReAct Agent** — A LangGraph prebuilt agent that reasons, calls tools, and synthesizes answers.

## Deploying the infrastructure on Azure

To deploy the required infrastructure on Azure, we use `Terraform`. The Terraform code provisions:
- An Azure Container Apps environment
- A serverless workload profile that uses GPU-enabled instances
- A container app running vLLM with the Gemma 4 31B IT model
- A session pool for Python REPL tool
- A container app for hosting MCP server for searching the web

Deploy these resources using the following `terraform` commands in the `555_llm_on_aca_gpu` directory:

In [ ]:
%terraform init
%terraform apply -auto-approve

## Get the LLM Endpoint

To consume the self-hosted model, we need to provide the base URL of our vLLM deployment and an API key (which can be any non-empty string since we disabled authentication in vLLM).

Retrieve the FQDN of the Gemma 4 model deployed on ACA from the Terraform output.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


Before building our agent, let's start with the basics. To consume an LLM model, OpenAI defines a standard API spec. It uses `openai` object with methods like `openai.chat.completions.create()`. vLLM implements this same API, so we can use it to connect to our self-hosted model.
Let's first install the OpenAI Python SDK, which we will use to call our vLLM model.

In [21]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Now we can consume the self-hosted model using OpenAI API.

In [ ]:
from openai import OpenAI

client = OpenAI(
    # base_url=f"http://4.165.31.28/v1",
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY"
)

response = client.chat.completions.create(
    model="google/gemma-4-31B-it",
    messages=[
        {"role": "user", "content": "Tell me about yourself."}
    ],
    max_tokens=512,
    temperature=1.0,
    stream=True
)

for chunk in response:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft Azure. It is designed to allow developers to build and deploy modern, microservices-based applications without having to manage the underlying infrastructure (like virtual machines or Kubernetes clusters).

At its core, ACA is built on top of **Azure Kubernetes Service (AKS)**, but it abstracts away all the complexity of Kubernetes. You get the power of orchestration (scaling, traffic splitting, etc.) without having to learn how to manage a K8s cluster.

Here is a detailed breakdown of its key features and how it works:

---

### 1. Key Core Features
*   **Serverless Scaling (KEDA):** ACA uses **KEDA** (Kubernetes Event-driven Autoscaling). This means your app can scale based on HTTP traffic, queue length (like Azure Service Bus), or custom events. Most importantly, it can **scale to zero**, meaning you don't pay when the app isn't being used.
*   **Built-in Dapr Integration:** It nati

Using `openai` API is enough to call the model, but it does not provide a great developer experience. So frameworks like `LangChain`, `Microsoft Agent Framework`, `Crew`, etc, can help to provide more rich features like memory management, conversation handling, tool integration and creating agents.

Next, we will use LangChain's `ChatOpenAI` class, which provides a more convenient interface and additional features for working with chat models.

In [26]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Simple Chat — No Tools

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the vLLM OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output. And note also how much similar the code is to calling OpenAI's API directly.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft. It allows you to deploy and scale applications and microservices without having to manage the underlying infrastructure (like virtual machines or Kubernetes clusters).

Think of it as a middle ground: it gives you the **power of Kubernetes** (orchestration, scaling, networking) but with the **simplicity of Serverless** (no cluster management, pay-as-you-go).

Here is a detailed breakdown of what it is and why it matters.

---

### 1. The Core Concept: "Serverless Kubernetes"
Under the hood, Azure Container Apps is built on **Azure Kubernetes Service (AKS)** and **KEDA** (Kubernetes Event-driven Autoscaling), but all the "Kubernetes complexity" is hidden from you.

*   **In AKS:** You have to manage nodes, upgrade versions, configure ingress controllers, and manage pods.
*   **In ACA:** You simply provide a container image (from Docker Hub or Azure Container Registry), and Azure handle

## 2. Define Tools

Create custom Python functions as tools using the `@tool` decorator. These will be available for the agent to call when needed.

In [37]:
from langchain_core.tools import tool
from random import randint


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {temp}°C."


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.

    Args:
        a: first number
        b: second number
    """
    return a * b


tools = [get_weather, multiply]
print("Registered tools:", [t.name for t in tools])

Registered tools: ['get_weather', 'multiply']


## 3. Tool Binding — Test Tool Calling

Bind the tools to the model and verify the LLM can decide when to call them.

In [5]:
model_with_tools = model.bind_tools(tools)

# This should NOT trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="Hi there!")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: Hello! How can I help you today?
Tool calls: []


In [38]:
# This SHOULD trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="What's the weather in Amsterdam?")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: 
Tool calls: [{'name': 'get_weather', 'args': {'location': 'Amsterdam'}, 'id': 'chatcmpl-tool-8d0306b5750ebde5', 'type': 'tool_call'}]


## 4. Create a ReAct Agent

Use LangGraph's `create_react_agent` to build an agent that can reason about when to call tools, execute them, and incorporate results into its response.

The agent implements the **ReAct** (Reasoning + Acting) pattern: it thinks about what to do, calls a tool if needed, observes the result, and repeats until it has a final answer.

In [7]:
from langchain.agents import create_agent

agent = create_agent(model, tools)

## 5. Run the Agent

### No tool needed — simple question

In [ ]:
response = agent.invoke({"messages": [HumanMessage(content="Tell me about yourself.")]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! What is Kubernetes?
================================== Ai Message ==================================

**Kubernetes** (often abbreviated as **K8s**) is an open-source platform designed to automate the deploying, scaling, and operating of application containers.

To understand Kubernetes, it helps to understand a few basic concepts first:

### 1. The Background: Containers
Before Kubernetes, developers started using **containers** (like Docker). A container bundles an application together with everything it needs to run (libraries, dependencies, settings). This ensures the app runs the same way whether it's on a developer's laptop or a production server.

### 2. The Problem: Container Sprawl
Running one or two containers is easy. But imagine a large company like Netflix or Spotify that runs *thousands* of containers across hundreds of different servers. Managing that manually is impossible. You have to 

### Tool call — weather query

The agent should recognize this requires the `get_weather` tool, call it, then respond with the result.

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather like in Amsterdam?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the weather like in Amsterdam?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-af58a8047b2cdb82)
 Call ID: chatcmpl-tool-af58a8047b2cdb82
  Args:
    location: Amsterdam
================================= Tool Message =================================
Name: get_weather

The weather in Amsterdam is cloudy with a high of 10°C.
================================== Ai Message ==================================

The weather in Amsterdam is currently cloudy with a high of 10°C.


### Tool call — multiply

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 7 multiplied by 13?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 7 multiplied by 13?
================================== Ai Message ==================================
Tool Calls:
  multiply (chatcmpl-tool-8d644489737ca673)
 Call ID: chatcmpl-tool-8d644489737ca673
  Args:
    a: 7
    b: 13
================================= Tool Message =================================
Name: multiply

91
================================== Ai Message ==================================

7 multiplied by 13 is 91.


## 6. Streaming

Stream the agent's step-by-step reasoning and responses in real time. Each step (LLM thinking, tool call, tool result, final answer) is printed as it occurs.

In [11]:
for step in agent.stream(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather in Paris?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-9743055666736ef1)
 Call ID: chatcmpl-tool-9743055666736ef1
  Args:
    location: Paris
================================= Tool Message =================================
Name: get_weather

The weather in Paris is sunny with a high of 12°C.
================================== Ai Message ==================================

The weather in Paris is sunny with a high of 12°C.


## More Resources

- [LangChain Tool Calling](https://python.langchain.com/docs/concepts/tool_calling/)
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) — the open standard for tool interoperability
- [Microsoft Learn MCP Server](https://learn.microsoft.com/api/mcp) — search Microsoft documentation via MCP